Here, we train scLEMBAS on the same split as Notebook 05B, but with different seeds to create an ensemble of learned models. We create an additional 4 per split in the 5-fold CV, giving a total of 25 models (5 of those originally trained in Notebook 05B and 20 from this split).

In [1]:
import os

import numpy as np

import torch 

import sys
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io
import scLEMBAS.utilities as utils

sys.path.insert(1, '../../.')
from McCauley_utils import initialize_mod_and_trainer, all_data

sys.path.insert(1, '../../../.') 
from notebook_utils import get_split

/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install

In [2]:
seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'
author = 'McCauley'

n_cores = 30
os.environ["OMP_NUM_THREADS"] = str(1)
os.environ["MKL_NUM_THREADS"] = str(1)
os.environ["OPENBLAS_NUM_THREADS"] = str(1)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(1)
os.environ["NUMEXPR_NUM_THREADS"] = str(1)

In [3]:
(sn_ppis, tf_adata, adata, expr, source_label, target_label, weight_label, 
 stimulation_label, inhibition_label, cat_col, pert_col, ctrl_pert) = all_data


In [17]:
# 1% of real
num_stochastic_edges = int(np.round(0.01*sn_ppis.shape[0]))

## 1% of real + fake
# num_real_edges = sn_ppis.shape[0]
# num_stochastic_edges = (0.01*num_real_edges)/0.99
# num_stochastic_edges = int(np.round*num_stochastic_edges)
print('We will add {} stochastic edges, representing 1% of the edges in the PKN'.format(num_stochastic_edges))



We will add 176 stochastic edges, representing 1% of the edges in the PKN


Train an ensemble of 5 models per fold (25 total), each model containing a new set of stochastic edges representing 1% of the total number of true edges: 

In [ ]:
n_ensembles_per_seed = 5
seed_multiplier = 21234

for fold in range(5):
    # -------------------- SINGLE-CELL MODELS --------------------
    fn_base = os.path.join(data_path, 'processed', '{}_fold{}'.format(author, fold))

    def write_model(mod, trainer, ensemble_idx):
        io.write_pickled_object(trainer, fn_base + 'pruning_trainer_actual_ensemble{}.pickle'.format(ensemble_idx))
        torch.save(mod.state_dict(), fn_base + 'pruning_model_actual_ensemble{}.pt'.format(ensemble_idx))
        del mod, trainer
        utils.clear_memory()

    for ensemble_idx in range(n_ensembles_per_seed):
        curr_seed = seed + ensemble_idx + 1 + (seed_multiplier * ensemble_idx * fold)        
        if os.path.isfile(fn_base + 'pruning_model_actual_ensemble{}.pt'.format(ensemble_idx)):
            continue
        mod, trainer = initialize_mod_and_trainer(
            fold = fold, 
            adversarial_penalty = True, 
            randomize = False, 
            num_stochastic_edges = num_stochastic_edges, 
            seed = curr_seed)
        mod = trainer.train_model(verbose = False)
        write_model(mod, trainer, ensemble_idx = ensemble_idx)
